<a id='top'></a>

![STScI Logo](../../../_static/stsci_header.png)

# WFC3/UVIS: Equalizing Amplifier Offsets in FLC/FLT Images
---
## Learning Goals
In this tutorial you will:

- Download data from MAST
- Build a segmentation map
- Measure the background in each quadrant
- Apply flat-field corrections and equalize amp bias levels

## Table of Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [Imports](#imports)
4. [Inputs](#inputs)
5. [Step 1: Build Source Segmentation Maps](#segmaps)
6. [Step 2: Measure Median Background Per Amp](#measure)
7. [Step 3: Compute Per-Amp Offsets](#offsets)
8. [Step 4: Apply the Flat-Field Correction and Equalize Amps](#correction)
9. [Step 5: Inspect the Results](#inspect)
10. [Step 6: Re-Drizzle the Corrected Images](#drizzle)
11. [Summary](#summary)



## 1. Introduction <a id='intro'></a>

In this tutorial, you will be guided through how to determine the amplifier-dependent bias offsets in a Wide Field Camera 3 (WFC3) UVIS channel image and equalize the bias levels in each amplifier (amp) to the average across the image.

The amp-offset correction workflow in this notebook follows the method from [Sunnquist (2019)](https://github.com/bsunnquist/uvis-skydarks) and uses much of the same code. An overview of the steps we will take to achieve this is as follows:

1. Build a source segmentation map to mask stars and galaxies before measuring sky backgrounds
2. Measure the background level in each amp quadrant (avoiding sources) and the mean over all amps 
3. Compute per-amp offsets relative to the overall mean background
4. Multiply these offsets by the flat field to account for sensitivity variations.
5. Subtract the resulting amp-by-amp corrections from the original `FLC`/`FLT` data.

If necessary, users should then re-drizzle their data using the corrected images to create new `DRZ`/`DRC` files after completing the above steps. 

The WFC3/UVIS detector has four amplifiers (A, B, C, D), each covering one quadrant of the 4Kx4K pixel field. Even after the standard `calwf3` pipeline reduction, small offsets to the bias level can persist between quadrants. These offsets are relatively constant across each quadrant, and sometimes show up as visible to the eye, with some amps brighter than others.


### WFC3/UVIS Amplifier Layout

The UVIS detector spans two CCDs (chips), each with two amps. The science data live in FITS extensions 1 (chip 2, lower) and 4 (chip 1, upper). 
```
┌──────────────────────┐
│   Amp A  │  Amp B    │  ← Extension 4, UVIS1
├──────────────────────┤
│   Amp C  │  Amp D    │  ← Extension 1, UVIS2
└──────────────────────┘
```

Within each extension, the left half corresponds to one amp and the right half to the other. 



## 2. Setup <a id='setup'></a>

### Environment

This notebook requires a working STScI Conda environment, `stenv`. If you do not have one, follow the [stenv installation guide](https://stenv.readthedocs.io/en/latest/) before proceeding.

This notebook requires users to install the packages listed in the requirements.txt file located in the notebook's sub-folder on the GitHub repository. Most of these packages should already exist in your `stenv` environment, but if you happen to be missing any required packages, they can be installed from the requirements.txt file with `pip install`.



## 3.  Imports <a id='imports'></a>

For this notebook we import:

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px;">Package</th>
      <th style="text-align: left; padding: 6px;">Purpose</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding: 6px;"><code>glob</code></td>
      <td style="padding: 6px;">File handling</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>matplotlib.pyplot</code></td>
      <td style="padding: 6px;">Displaying images and visualizing offsets</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>numpy</code></td>
      <td style="padding: 6px;">Handling arrays</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>os</code></td>
      <td style="padding: 6px;">System commands</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>shutil</code></td>
      <td style="padding: 6px;">File and directory clean up</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>astropy.io.fits</code></td>
      <td style="padding: 6px;">Reading and modifying FITS files</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>astropy.convolution</code></td>
      <td style="padding: 6px;">Tools to smooth the background of images</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>astropy.stats</code></td>
      <td style="padding: 6px;">Tools to smooth the background of images</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>astroquery.mast.Observations</code></td>
      <td style="padding: 6px;">Downloading FLC and DRC data from MAST</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>photutils.segmentation</code></td>
      <td style="padding: 6px;">Creating source masks</td>
    </tr>
    <tr>
      <td style="padding: 6px;"><code>drizzlepac</code></td>
      <td style="padding: 6px;">Aligning and re-combining images</td>
    </tr>
  </tbody>
</table>


In [ ]:
import glob
import os
import shutil

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from astropy.io import fits
from astropy.convolution import convolve, Gaussian2DKernel
from astropy.stats import gaussian_fwhm_to_sigma
from astroquery.mast import Observations
from photutils.segmentation import detect_sources, detect_threshold
from drizzlepac import tweakreg, astrodrizzle
from drizzlepac.processInput import getMdriztabPars


## 4. Inputs <a id='inputs'></a>


### Downloading an Example Image 

For the purposes of this tutorial, we will download a set of WFC3/UVIS images with clear amp-dependent offsets and work through the steps to correct the offsets. Alternatively, you could download your own data to run through this pipeline! We recommend working through the example data first. 


We will use [Astroquery](https://astroquery.readthedocs.io/en/latest/) to
retrieve files. We query for data from the Mikulski Archive for Space Telescopes (MAST) as follows:

In [ ]:
# Define the dataset you want. In this case, we are using the obsID for the set of two example exposures
obsID = 'ifcpc6030'
# Query observations by obs_id
obs_table = Observations.query_criteria(obs_id=obsID)

# Get list of data products
data_products = Observations.get_product_list(obs_table)
data_products

In [ ]:
# Filter for calibrated science products (in this case, FLC and DRC files from CALWF3)
filtered_products = Observations.filter_products(
    data_products,
    productSubGroupDescription=["FLC", "DRC",],
    project='CALWF3' 
)

filtered_products

In [ ]:
# Download
manifest = Observations.download_products(
    filtered_products,
    mrp_only=False,    
    download_dir="./"  # setting the download path to the current working directory
)

print(manifest)

Please note: If you are working on a set of images, it is important to place all FLC (or FLT) files for a **single filter** in a working directory and set the path. Code is provided below to do this. All files must use the same filter because the flat-field correction step relies on the `PFLTFILE` keyword in the primary header, which is filter-dependent.

In [ ]:
# Directory where files were downloaded
download_dir = "mastDownload"

# Loop over all FITS files
for root, dirs, files in os.walk(download_dir):
    for file in files:
        if file.endswith("flc.fits"):
            filepath = os.path.join(root, file)

            try:
                # Read FITS header
                with fits.open(filepath) as hdul:
                    hdr = hdul[0].header

                # Get filter name
                filt = hdr.get("FILTER", None)

                if filt is None or filt == "":
                    print(f"Skipping {file}: no filter found")
                    continue

                # Create destination directory (one level below cwd)
                dest_dir = os.path.join(os.getcwd(), filt)
                os.makedirs(dest_dir, exist_ok=True)

                # Move file
                dest_path = os.path.join(dest_dir, file)
                shutil.move(filepath, dest_path)

                print(f"Moved {file} → {dest_dir}")

            except Exception as e:
                print(f"Error processing {file}: {e}")

In [ ]:
# Filter-specific directory containing the FLC/FLT files (in this case, the exposures we care about are in './F336M/')
filter_dir = './F336W'

os.chdir(filter_dir)
# Glob to select your science files
files = sorted(glob.glob('*flc.fits'))

print(f'Found {len(files)} input file(s):')
for f in files:
    print(f'  {f}')

Let's take a look at our images. Below, we plot the science data of our two example images. Using a narrow stretch, we can see the amp-to-amp background offsets by eye, with some quadrants appearing darker/lighter than others.

In [ ]:
for fname in files:

    # Load both chips
    chip2 = fits.getdata(fname, ext=1)  # UVIS2
    chip1 = fits.getdata(fname, ext=4)  # UVIS1


    # Stack vertically 
    full = np.vstack([chip2, chip1])

    # Tight stretch to highlight amp offsets
    vmin, vmax = np.percentile(full, [45, 55])

    plt.figure(figsize=(8, 10))
    plt.imshow(full, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    plt.colorbar(label="Electrons ($e^{-}$)", shrink=0.6,)
    plt.title(fname)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.show()

Lastly, we need to set our `iref` path to point to a directory containing WFC3 reference files. We will use the Calibration Reference Data System (CRDS), which automatically downloads the right reference files on demand.

In [ ]:
crds_path = os.path.expanduser("~") + "/crds_cache"
os.environ["CRDS_PATH"] = crds_path
os.environ["CRDS_SERVER_URL"] = "https://hst-crds.stsci.edu"
os.environ["iref"] = os.path.join(crds_path, "references/hst/wfc3/")


## 5. Step 1 — Build Source Segmentation Maps <a id='segmaps'></a>

Before we can measure the background sky level in each amp quadrant, we need to identify and mask any pixels that belong to real sources (stars, galaxies, etc.). We do this using a segmentation map, in which pixels belonging to a detected source are labeled with a nonzero integer, and pure-sky pixels are labeled 0.

We use the photutils package to detect sources and mask them for our image. We mask only sources that span 3 or more pixels and are at least 1 standard deviation per pixel above the background. A 3 sigma Gaussian smoothing kernel is applied before source detection to average out single bright pixels from non-source artifacts such as hot pixels or cosmic rays. We use a low signal-to-noise-ratio (SNR) threshold of 1 standard deviation because we would rather over-mask a few sky pixels than accidentally include a faint source wing in our background estimate, which would bias the amp-level measurements.

The segmentation maps ("segmaps") are written out as `{filename}_seg_ext_{1|4}.fits` so they can be reused in later steps without repeating the detection.

In [ ]:
def make_segmap(f, smoothing_sigma=3, threshold_sigma=1.0, source_pixel_span=3, overwrite=True):
    """
    Build a source segmentation map for each science extension of a UVIS file.

    Parameters
    ----------
    f : str
        Path to the FLC/FLT FITS file.
    smoothing_sigma : float, optional
        FWHM in pixels of the Gaussian smoothing kernel applied before source
        detection to average out single-pixel artifacts such as hot pixels or
        cosmic rays. Default is 3.
    threshold_sigma : float, optional
        The sigma (signal-to-noise) threshold above which a pixel is considered
        part of a source. Default is 1.0.
    source_pixel_span : int, optional
        Minimum number of contiguous pixels required for a detection to be
        considered a source. Default is 3.
    overwrite : bool, optional
        If True, regenerate the segmap even if it already exists on disk.
        Default is True.

    Outputs
    -------
    {f}_seg_ext_1.fits, {f}_seg_ext_4.fits
        Segmentation maps for each SCI extension of the FITS file.
    """
    sigma = smoothing_sigma * gaussian_fwhm_to_sigma  
    kernel = Gaussian2DKernel(sigma, x_size=3, y_size=3)
    kernel.normalize()

    for ext in [1, 4]:
        outfile = f.replace('.fits', f'_seg_ext_{ext}.fits')
        if os.path.exists(outfile) and not overwrite:
            print(f'  Segmap already exists, skipping: {outfile}')
            continue

        data = fits.getdata(f, ext)
        threshold = detect_threshold(data, nsigma=threshold_sigma)
        convolved_data = convolve(data, kernel, normalize_kernel=True)
        segm = detect_sources(convolved_data, threshold, npixels=source_pixel_span)

        fits.writeto(outfile, segm.data.astype(np.int32), overwrite=True)
        print(f'  Written: {outfile}')

In [ ]:
print('Building segmentation maps...\n')
for f in files:
    print(f'Processing {f}')
    make_segmap(f, overwrite=False)
print('\nDone.')

### Inspect the segmentation maps

We want to visually verify that the segmaps capture the sources without over-masking too much sky. In the plot below, we show our original science data on the left, and the segmentation map on the right with masked (source) pixels shown in yellow and sky pixels in dark purple/blue. In our example images, the brightest source falls on the upper chip (UVIS1, SCI extension 4). 

In [ ]:
for fname in files:
    fname
    # Load both chips
    data_ext1 = fits.getdata(fname, 1)  # UVIS2
    data_ext4 = fits.getdata(fname, 4)  # UVIS1

    segmap_ext1 = fits.getdata(fname.replace('.fits', '_seg_ext_1.fits'))
    segmap_ext4 = fits.getdata(fname.replace('.fits', '_seg_ext_4.fits'))

    # Stack vertically
    data_full = np.vstack([data_ext1, data_ext4])
    segmap_full = np.vstack([segmap_ext1, segmap_ext4])

    vmin, vmax = np.percentile(data_full, [45, 55])

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))

    axes[0].imshow(data_full, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    axes[0].set_title('Science image (both chips)', fontsize=13)
    axes[0].set_xlabel('X pixel')
    axes[0].set_ylabel('Y pixel')

    axes[1].imshow(segmap_full > 0, origin='lower', cmap='viridis')
    axes[1].set_title('Source mask (yellow = masked)', fontsize=13)
    axes[1].set_xlabel('X pixel')

    plt.suptitle(os.path.basename(fname), fontsize=12)
    plt.tight_layout()
    plt.show()


## 6. Step 2 — Measure Median Background Per Amp <a id='measure'></a>

With sources masked, we can now measure a robust background level for each of the four amp quadrants. We use the median rather than the mean because it is less sensitive to residual contamination from faint sources or bad pixels.

Each UVIS science extension covers two amps. We split each extension exactly in half along the column axis to grab each amp individually. 

Source pixels (segmap > 0) are set to `NaN` before the split so that `np.nanmedian` automatically ignores them.

In [ ]:
def measure_amp_levels(f):
    """
    Measure the median sky background in each of the four UVIS amp quadrants.

    Parameters
    ----------
    f : str
        Path to the FLC/FLT FITS file.

    Returns
    -------
    amp_levels : list of float
        [median_C, median_D, median_A, median_B]  (electrons)
    amp_average : float
        Mean of all four amp medians.
    """
    amp_levels = []

    for ext in [1, 4]:
        data = fits.getdata(f, ext).astype(float)
        segmap = fits.getdata(f.replace('.fits', f'_seg_ext_{ext}.fits'))

        # Mask sources
        data[segmap > 0] = np.nan

        # Split into left (amp A/C) and right (amp B/D) halves
        left, right = np.split(data, 2, axis=1)

        amp_levels.append(np.nanmedian(left))
        amp_levels.append(np.nanmedian(right))

    amp_average = np.nanmean(amp_levels)
    return amp_levels, amp_average

In [ ]:
print(f'{"File":<35}  {"Amp A":>8}  {"Amp B":>8}  {"Amp C":>8}  {"Amp D":>8}  {"Mean":>8}')
print('-' * 85)

all_levels = {}
for f in files: # in our example case we only have two files
    levels, avg = measure_amp_levels(f)
    all_levels[f] = (levels, avg)
    C, D, A, B = levels  # extension 1 (UVIS2, amps C & D) are appended to the list first
    print(f'{os.path.basename(f):<35}  {A:8.4f}  {B:8.4f}  {C:8.4f}  {D:8.4f}  {avg:8.4f}')


## 7. Step 3 — Compute the Per-Amp Offsets <a id='offsets'></a>

The offset for each amplifier quadrant is the deviation of its median background from the overall mean (the mean of the per-amp medians).

A positive offset means that the amp reads slightly high relative to the average, and we will want to subtract the offset to correct for it. After the correction, the four quadrant levels should all be approximately equal to the overall mean across all four amps.

Below, we plot the per-amp offsets from the mean before equalization as a bar chart for each image and explicitly print the offset information.

In [ ]:
# Visualise the offsets for each file as a bar chart
print('Per-amp offsets from mean before equalization (e-)')
print(f'{"File":<30} {"A":>8}  {"B":>8}  {"C":>8}  {"D":>8}')
print('-' * 85)

amp_labels = ['A (ext4-left)', 'B (ext4-right)', 'C (ext1-left)', 'D (ext1-right)']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, len(files), figsize=(4 * len(files), 4), sharey=True)
if len(files) == 1:
    axes = [axes]

for ax, f in zip(axes, files):
    levels, avg = all_levels[f]
    offsets = [lv - avg for lv in levels]
    reordered = [offsets[i] for i in [2, 3, 0, 1]] 
    print(f'{os.path.basename(fname):<30}  {reordered[0]:8.4f}  {reordered[1]:8.4f} {reordered[2]:8.4f}  {reordered[3]:8.4f}')
    bars = ax.bar(amp_labels, reordered, color=colors, alpha=0.85, edgecolor='black', linewidth=0.6)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(os.path.basename(f), fontsize=9)
    ax.set_xticklabels(amp_labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Offset from mean (e⁻)', fontsize=9)

fig.suptitle('Per-amp offsets before equalization', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()


## 8. Step 4 — Apply the Flat-Field Correction and Equalize Amps <a id='correction'></a>


The FLC/FLT data have already been "flat-fielded" (divided by the flat-field image). The residual amp offsets, therefore, have also been modulated by the flat. To correct for this, we multiply the amp-to-amp scalar offset arrays by the flat before subtracting from the science data. This ensures the subtracted correction has exactly the same pixel-level shape as the bias offset would have had before flat-fielding, producing a cleaner result.

The flat-field file is identified from the `PFLTFILE` keyword in the primary header and loaded from the `$iref` reference file directory.

In [ ]:
def equalize_amps(f, multiply_flat=True):
    """
    Equalize the four UVIS amp levels in an FLC/FLT image.

    For each amp quadrant the median background (computed with sources masked)
    is compared to the mean of all four quadrants.  The difference
    (optionally multiplied by the flat field) is subtracted from the
    original science data.

    Parameters
    ----------
    f : str
        Path to the FLC/FLT FITS file.  Segmaps must already exist (Step 1).
    multiply_flat : bool
        If True (recommended), scale the offset by the flat field before
        subtracting so that the correction accounts for sensitivity variations
        across each quadrant.

    Outputs
    -------
    {f}_eq.fits
        A copy of the input file with equalized amp levels written to SCI
        extensions 1 and 4.
    """
    outfile = f.replace('.fits', '_eq.fits')
    if os.path.isfile(outfile):
        print(f'  Output already exists, skipping: {outfile}')
        return outfile

    h = fits.open(f)

    # Locate the flat-field reference file if needed
    if multiply_flat:
        pflt_key = h[0].header['PFLTFILE'].replace('iref$', '')
        iref = os.environ['iref']
        os.system(f'crds sync --hst --files {pflt_key} --output-dir {iref}')
        flat_file = os.path.join(os.environ['iref'], pflt_key)
        print(f'  Using flat field: {os.path.basename(flat_file)}')

    #  Collect the median background level in each of the 4 amp quadrants 
    amp_levels = []
    for ext in [1, 4]:
        data = h[ext].data.astype(float).copy()
        segmap = fits.getdata(f.replace('.fits', f'_seg_ext_{ext}.fits'))
        data[segmap > 0] = np.nan          # mask sources
        left, right = np.split(data, 2, axis=1)
        amp_levels.append(np.nanmedian(left))
        amp_levels.append(np.nanmedian(right))

    amp_average = np.nanmean(amp_levels)

    print(f'  Amp medians  : A={amp_levels[2]:.4f}  B={amp_levels[3]:.4f}  '
          f'C={amp_levels[0]:.4f}  D={amp_levels[1]:.4f}')
    print(f'  Overall mean   : {amp_average:.4f}')
    offsets = [lv - amp_average for lv in amp_levels]
    print(f'  Offsets (C,D,A,B): {[f"{o:.4f}" for o in offsets]}')

    # Subtract the offset from the original data 

    for ext in [1, 4]:
        data_orig = h[ext].data.astype(float).copy()
        left_orig, right_orig = np.split(data_orig, 2, axis=1)

        # Source-masked copy for measuring 
        data_masked = data_orig.copy()
        segmap = fits.getdata(f.replace('.fits', f'_seg_ext_{ext}.fits'))
        data_masked[segmap > 0] = np.nan
        left_m, right_m = np.split(data_masked, 2, axis=1)

        if multiply_flat:
            flat = fits.getdata(flat_file, ext).astype(float)
            flat_left, flat_right = np.split(flat, 2, axis=1)
            correction_left = (np.nanmedian(left_m) - amp_average) * flat_left
            correction_right = (np.nanmedian(right_m) - amp_average) * flat_right
        else:
            correction_left = np.nanmedian(left_m) - amp_average
            correction_right = np.nanmedian(right_m) - amp_average

        left_new = left_orig - correction_left
        right_new = right_orig - correction_right

        h[ext].data = np.concatenate([left_new, right_new], axis=1).astype('float32')

    h.writeto(outfile, overwrite=True)
    h.close()
    print(f'  Written: {outfile}')
    return outfile

In [ ]:
print('Equalizing amp levels...\n')
eq_files = []
for f in files:
    print(f'Processing {f}')
    outfile = equalize_amps(f, multiply_flat=True)
    eq_files.append(outfile)
    print()
print('Done.')


## 9. Step 5 — Inspect the Results <a id='inspect'></a>

We now compare the per-amp medians **before** and **after** equalization to verify that the correction worked as expected. After equalization all four quadrant medians should be within a small fraction of an electron of each other.

In [ ]:
print('Amp medians before and after equalization\n')
print(f'{"File":<30}  {"State":<8}  {"A":>8}  {"B":>8}  {"C":>8}  {"D":>8}  {"Spread":>8}')
print('-' * 90)

for f, f_eq in zip(files, eq_files):
    for label, fname in [("before", f), ("after", f_eq)]:
        seg1 = fname.replace('.fits', '_seg_ext_1.fits')
        seg4 = fname.replace('.fits', '_seg_ext_4.fits')

        if not all(os.path.exists(f) for f in [fname, seg1, seg4]):
            make_segmap(fname, overwrite=False)
        levels, avg = measure_amp_levels(fname)
        C, D, A, B = levels
        spread = max(levels) - min(levels)
        print(f'{os.path.basename(fname):<30}  {label:<8}  {A:8.4f}  {B:8.4f}  '
              f'{C:8.4f}  {D:8.4f}  {spread:8.4f}')
    print()

Below, we compare our original science data (left) and post-equalization science data (right) for each image. The science images after equalization no longer exhibit noticeable amp-to-amp brightness differences, as their background levels differ by only a fraction of an electron.

In [ ]:
# Full image comparison before and after equalization
for fname in files:
    example_file = fname
    example_eq_file = example_file.replace('.fits', '_eq.fits')

    # Load both chips
    data_ext1 = fits.getdata(example_file, 1)  # UVIS2
    data_ext4 = fits.getdata(example_file, 4)  # UVIS1

    data_eq_ext1 = fits.getdata(example_eq_file, 1)
    data_eq_ext4 = fits.getdata(example_eq_file, 4)

    # Stack vertically
    data_full = np.vstack([data_ext1, data_ext4])
    data_eq_full = np.vstack([data_eq_ext1, data_eq_ext4])

    vmin, vmax = np.percentile(data_full, [49, 51])

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))

    axes[0].imshow(data_full, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    axes[0].set_title('Science image before equalization (both chips)', fontsize=13)
    axes[0].set_xlabel('X pixel')
    axes[0].set_ylabel('Y pixel')

    axes[1].imshow(data_eq_full, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    axes[1].set_title('Science image after equalization (both chips)', fontsize=13)
    axes[1].set_xlabel('X pixel')
    axes[1].set_ylabel('Y pixel')

    plt.suptitle(os.path.basename(example_file), fontsize=12)
    plt.tight_layout()
    plt.show()


## 10. Step 6 — Re-Drizzle the Corrected Images <a id='drizzle'></a>
If you downloaded a pre-drizzled `DRZ` or `DRC` product and only applied amp equalization to the underlying FLT/FLCs, you must re-drizzle, as the downloaded `DRZ`/`DRC` was created from the uncorrected frames and will not reflect your changes.  If you are starting fresh from `FLT`/`FLC` and have not yet drizzled, simply run AstroDrizzle on the `*_eq.fits` files directly.
The corrected `*_eq.fits` images can be combined into a drizzled image using `AstroDrizzle` from the `drizzlepac` package.



We provide a worked example using our 2 example images below; see the [DrizzlePac handbook](https://drizzlepac.readthedocs.io) and the [DrizzlePac notebooks](https://spacetelescope.github.io/hst_notebooks/notebooks/DrizzlePac/README.html) for full guidance on parameters.


In [ ]:
tweakreg.TweakReg(
    eq_files,
    imagefindcfg={"threshold": 200, "conv_width": 6},
    shiftfile=True,
    outshifts="shift_equalized_flc_test.txt",
     wcsname = 'TWEAK1',
    updatehdr=True,
    interactive=False,
    ylimit=0.4,
    searchrad=0.15,
    tolerance=1,
)

In [ ]:
# The following lines of code find and download the MDRIZTAB reference file which can be used as drizzle parameters
mdz = fits.getval(eq_files[0], 'MDRIZTAB', ext=0).split('$')[1]
print('Searching for the MDRIZTAB file:', mdz)
os.system(f'crds sync --hst --files {mdz} --output-dir {os.environ["iref"]}')

In [ ]:
def get_vals_from_mdriztab(
    input_files,
    kw_list=[
        "driz_sep_bits",
        "combine_type",
        "driz_cr_snr",
        "driz_cr_scale",
        "final_bits",
    ],
):
    """Get only selected parameters from. the MDRIZTAB"""
    mdriz_dict = getMdriztabPars(input_files)

    requested_params = {}

    print("Outputting the following parameters:")
    for k in kw_list:
        requested_params[k] = mdriz_dict[k]
        print(k, mdriz_dict[k])

    return requested_params

selected_params = get_vals_from_mdriztab(eq_files)

In [ ]:
output_name = 'ifcpc6030_equalized'   # prefix for output DRZ file
astrodrizzle.AstroDrizzle(
    eq_files,
    output=output_name,
    **selected_params,
    preserve=False,
    clean=True,
    build=True,
    context=False,
    skymethod="match",
    runfile="f336w_eq_driz.log"
)

Finally, we plot the `DRC` science data (left) and new `DRZ` science data post-equalization (right) created by combining our images with Astrodrizzle. The new `DRZ` no longer has amp-to-amp brightness differences. 

In [ ]:
# Full drizzled image comparison before and after equalization

example_file = '../mastDownload/HST/ifcpc6030/ifcpc6030_drc.fits' 
example_eq_file = './ifcpc6030_equalized_drz.fits'

# Load drizzled images
data_full = fits.getdata(example_file, 1) 

data_eq_full = fits.getdata(example_eq_file, 1)


vmin, vmax = np.nanpercentile(data_full, [5, 95])

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

axes[0].imshow(data_full, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('Science image before equalization (both chips)', fontsize=13)
axes[0].set_xlabel('X pixel')
axes[0].set_ylabel('Y pixel')

axes[1].imshow(data_eq_full, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('Science image after equalization(both chips)', fontsize=13)
axes[1].set_xlabel('X pixel')
axes[1].set_ylabel('Y pixel')

plt.suptitle(os.path.basename(example_file), fontsize=12)
plt.tight_layout()
plt.show()

## 11. Summary <a id='summary'></a>
Thank you for going through this notebook. You should now be familiar with the workflow to correct amp-dependent bias offsets in your WFC3 UVIS images, including the following steps:

- Downloading data from MAST.
- Measuring bias levels in each amp and computing individual offsets.
- Accounting for the flat-field.
- Equalizing amp-dependent bias levels across the image and redrizzling FLT/FLCs.



### Additional Resources
Below are some additional resources that may be helpful. Please send any questions through the [HST Help Desk](https://stsci.service-now.com/hst).

- Sunnquist, B. 2019, *UVIS Sky Darks*, [GitHub repository](https://github.com/bsunnquist/uvis-skydarks)
- [DrizzlePac documentation](https://drizzlepac.readthedocs.io)
- [STScI HST Notebooks repository](https://spacetelescope.github.io/hst_notebooks/)

## About this Notebook

**Author:** Anne O'Connor, WFC3 Team

**Created** 2027-05-27



## Citations

This notebook was developed based on the 
amp-equalization code and routines in:

> Sunnquist, B. (2019). *UVIS Sky Darks*. GitHub.
> https://github.com/bsunnquist/uvis-skydarks

If you use this notebook in published work, please cite the original 
Sunnquist (2019) repository and any relevant Python packages such as `astropy, astroquery, drizzlepac, matplotlib,` or `numpy`. Follow these links for more information about citing various packages.

* [Citing `astropy`](https://www.astropy.org/acknowledging.html)
* [Citing `astroquery`](https://github.com/astropy/astroquery/blob/main/astroquery/CITATION)
* [Citing `drizzlepac`](https://zenodo.org/records/3743274)
* [Citing `matplotlib`](https://matplotlib.org/stable/project/citing.html)
* [Citing `numpy`](https://numpy.org/citing-numpy/)

[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/hst_notebooks/main/assets/stsci_pri_combo_mark_horizonal_white_bkgd.png" alt="Space Telescope Logo" width="200px\"/>